# Analysing pedigrees: PedNet, PedGen and PedInterp

This notebook works with three more packages: **PedNet**, which analyses a pedigree as a graph and weighs up the mode
of inheritance and candidate variants; **PedGen**, which simulates families; and **PedInterp**, which reads a picture
of a pedigree. They need NumPy, and PedGen and PedInterp need pandas too, and PedInterp OpenCV, so the workbench loads
them only when they are first used. The first cell that imports one of them takes longer, because it downloads it,
with what it needs: about 4 MB for PedNet, and about 21 MB in all once PedGen and PedInterp are loaded too. Your
browser keeps them for next time.

Run the cells in order from the top: click a cell and press **Shift+Enter**. The same work is in the **Tools** tab,
under *Analyse and simulate*, and in the terminal, as the commands `pednet`, `pedgen` and `pedinterp`.

## A family as a graph

PedNet reads a PED file into a pedigree for each family. A pedigree is a directed acyclic graph, with an edge from
each parent to each child, and a marriage between relatives closes a loop in it. In the Okafor family, the parents of
the twin boys with hearing loss are first cousins. (From an MPED file, PedNet takes the first condition.)

In [ ]:
import pednet

okafor = pednet.read_ped("/data/examples/okafor.mped")["FAM1"]
print(len(okafor.people), "people; affected:", okafor.affected())
for loop in pednet.loops(okafor):
    print("a loop, from", loop.kind + ":", ", ".join(loop.people))

## Kinship and inbreeding

The kinship coefficient of two people is the chance that a copy of a gene taken at random from one of them and a copy
taken from the other are the same copy, inherited from an ancestor they share. A child's inbreeding coefficient is
the kinship of their parents, so the children of first cousins have 1/16.

In [ ]:
inbred = {pid: f for pid, f in pednet.inbreeding(okafor).items() if f > 0}
print("inbred:", inbred)
for a, b in [("IV-3", "IV-4"), ("IV-3", "II-3"), ("III-2", "III-3")]:
    r = pednet.relationship(okafor, a, b)
    print(f"{b} is the {r.name} of {a}: a relative of degree {r.degree}, kinship {r.kinship:.4f}")
# everyone's, as a matrix: a person's kinship with themself is (1 + F)/2
ids, phi = pednet.kinship_matrix(okafor)
print("IV-3 with himself:", phi[ids.index("IV-3"), ids.index("IV-3")])

## The mode of inheritance

`infer_mode` works out the likelihood of who is affected under each mode of inheritance exactly, by treating the
pedigree as a Bayesian network of genotypes, and turns the likelihoods into probabilities. In the Lynch family, bowel
cancer passes from father to son, and the answer is clear. Two affected brothers whose parents are cousins point to a
recessive condition, but not to whether it is autosomal or X-linked.

In [ ]:
lynch = pednet.read_ped("/data/examples/lynch.ped")["FAM1"]
print(pednet.infer_mode(lynch))
print()
print(pednet.infer_mode(okafor))

## Candidate variants

`rank_variants` scores each candidate variant by how well it co-segregates with the condition. The score is a Bayes
factor, and its strength is given as evidence for co-segregation (PP1) or against it (BS4), under the ACMG/AMP
guidelines. These are PedNet's own examples: a family with a dominant condition, and six variants found by
sequencing some of them.

In [ ]:
family = pednet.read_ped("/data/examples/pednet/dominant.ped")["FAM1"]
variants = pednet.read_vcf("/data/examples/pednet/dominant.vcf", samples=family.ids)
for s in pednet.rank_variants(family, variants):
    print(s.rank, s.id, round(s.log10_bf, 2), s.evidence, s.best_mode, s.notes)

## Simulating families with PedGen

PedGen makes up a family by Mendelian inheritance from an affected founder, as a table, and with a seed it makes the
same family each time. PedNet reads the table as it is.

In [ ]:
from PedGen import pedigree_generator, gen_PED_export

sim = pedigree_generator(max_children=3, FamilyID="SIM1", mode="AD", generation_count=4, seed=4)
print(len(sim), "people,", (sim["Phenotype"] == 2).sum(), "affected")
print(pednet.infer_mode(pednet.pedigree_from_rows(sim)))
print("wrote", gen_PED_export(sim, "/home/user/results"))

PedNet takes a condition to be rare: its recessive model gives the allele a frequency of 1 in 1,000. The partners
who marry into a family that PedGen simulates carry the allele at the frequency you give, 0.1 unless you say
otherwise, so about one in five of them is a carrier. In a recessive family that PedGen makes, the affected children
of carriers who married in are very unlikely under PedNet's model, and it may favour another mode. Given PedGen's
frequency, it weighs the family up as PedGen made it.

In [ ]:
rec = pednet.pedigree_from_rows(pedigree_generator(max_children=3, FamilyID="SIM2", mode="AR", generation_count=4, seed=34))
print("a rare allele:", pednet.infer_mode(rec).call)
print("PedGen's frequency:", pednet.infer_mode(rec, allele_frequency={"AR": 0.1}).call)

## Reading a picture with PedInterp

PedInterp reads a pedigree from a picture of one, drawn in black on white. The labels can't be read in a browser,
which has no Tesseract, so the people are numbered by generation instead (I-1, I-2, II-1 …), as Pedigree Studio
numbers them. The picture of the Lynch family was drawn where the Studio draws it, so the numbers are the ones in the
Lynch PED file, and the two tables are the same.

In [ ]:
import pandas as pd
from PedInterp import pedigree_processing

read = pedigree_processing("LYNCH", image_path="/data/examples/lynch.png", ocr=None)
print(read.head(6).to_string(index=False))
columns = ["FamilyID", "IndividualID", "PaternalID", "MaternalID", "Sex", "Phenotype"]
ped = pd.read_csv("/data/examples/lynch.ped", sep="\t", header=None, names=columns, dtype=str)
print("the same as lynch.ped:", read[columns[1:]].astype(str).values.tolist() == ped[columns[1:]].values.tolist())

## The commands, from a cell

The same work as commands: a line that starts with `!` runs one, as the terminal would.

In [ ]:
!pednet mode /data/examples/okafor.mped --method patterns
!pedgen --mode AD --generations 3 --seed 5 --family SIM3 -d ~/results
!pedinterp /data/examples/lynch.png --labels none -q -d ~/results